In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
from skimage.morphology import convex_hull_image


In [ ]:

def fit_ellipse(mask):
    # Foreground pixel coordinates
    y, x = np.nonzero(mask)

    if len(x) < 3:
        raise ValueError("Mask must contain at least 3 foreground pixels.")

    # Center of mass
    cx = x.mean()
    cy = y.mean()

    # Centered coordinates
    dx = x - cx
    dy = y - cy

    # Covariance / second moment matrix
    cov = np.cov(np.stack([dx, dy]))

    # Eigenvalues/eigenvectors
    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    # Sort: largest eigenvalue = major axis
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # Axis directions
    major_axis = eigenvectors[:, 0]
    minor_axis = eigenvectors[:, 1]

    # Orientation of major axis
    angle = np.arctan2(major_axis[1], major_axis[0])

    # Convert to degrees
    angle_deg = np.degrees(angle)

    # Ellipse semi-axis lengths
    # For a uniform filled ellipse:
    # variance = semi_axis^2 / 4
    semi_major = 2 * np.sqrt(eigenvalues[0])
    semi_minor = 2 * np.sqrt(eigenvalues[1])

    return {
        "center": (cx, cy),
        "semi_major": semi_major,
        "semi_minor": semi_minor,
        "major_axis": major_axis,
        "minor_axis": minor_axis,
        "angle_rad": angle,
        "angle_deg": angle_deg,
    }



def plot_mask_ellipse(mask, ellipse):

    plt.figure()

    # Mask
    plt.imshow(mask, origin="upper")
    
    # Ellipse and center
    theta = np.linspace(0, 2*np.pi, 200)
    a = ellipse["semi_major"]
    b = ellipse["semi_minor"]
    cx, cy = ellipse["center"]
    angle = ellipse["angle_rad"]
    R = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle),  np.cos(angle)]
    ])
    ellipse_xy = R @ np.vstack([
        a * np.cos(theta),
        b * np.sin(theta)
    ])
    ellipse_x = ellipse_xy[0] + cx
    ellipse_y = ellipse_xy[1] + cy
    plt.plot(ellipse_x, ellipse_y)
    plt.scatter(cx, cy, c="yellow")
    
    # Major axis
    x = [
        ellipse['center'][0] - ellipse['semi_major']*ellipse['major_axis'][0],
        ellipse['center'][0] + ellipse['semi_major']*ellipse['major_axis'][0]
    ]
    y = [
        ellipse['center'][1] - ellipse['semi_major']*ellipse['major_axis'][1],
        ellipse['center'][1] + ellipse['semi_major']*ellipse['major_axis'][1]
    ]
    plt.plot(x,y)
    
    # Minor axis
    x = [
        ellipse['center'][0] - ellipse['semi_minor']*ellipse['minor_axis'][0],
        ellipse['center'][0] + ellipse['semi_minor']*ellipse['minor_axis'][0]
    ]
    y = [
        ellipse['center'][1] - ellipse['semi_minor']*ellipse['minor_axis'][1],
        ellipse['center'][1] + ellipse['semi_minor']*ellipse['minor_axis'][1]
    ]
    plt.plot(x,y)
    
    # Zoom box
    margin = 50
    plt.ylim(min(np.where(mask)[0])-margin, max(np.where(mask)[0])+margin)
    plt.xlim(min(np.where(mask)[1])-margin, max(np.where(mask)[1])+margin)
    
    # plt.axis("equal")
    plt.show()

In [ ]:
# Input
folder_masks = '/tmp/kiwibes2026/mask'


In [ ]:
files_mask = glob.glob(f'{folder_masks}/*npy')
len(files_mask)

In [ ]:
for file_mask in files_mask:
    print(file_mask)
    
    # Load
    mask = np.load(file_mask)
    
    # Fit ellipse
    ellipse = fit_ellipse(mask)
    
    # Vis
    plot_mask_ellipse(mask, ellipse)